In [1]:
import pickle 
import mlflow 
import mlflow.sklearn


import pandas as pd 

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


from sklearn.pipeline import make_pipeline

In [23]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = ""


In [2]:
import mlflow 

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration1")

<Experiment: artifact_location='s3://s3-bucket-default-mlflow/2', creation_time=1759829720602, experiment_id='2', last_update_time=1759829720602, lifecycle_stage='active', name='green-taxi-duration1', tags={}>

In [3]:
def read_dataframe(filename:str):
    df=pd.read_parquet(filename)
    
    df['duration']=df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical=['PULocationID', 'DOLocationID']
    df[categorical]=df[categorical].astype(str)
    return df


In [4]:
def prepare_dict(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical=["PU_DO"]
    numerical=["trip_distance"]
    dicts=df[categorical + numerical].to_dict(orient='records')
    return dicts

In [5]:
df_train=read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
df_val=read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")

target="duration"
y_train=df_train[target].values
y_val=df_val[target].values

dict_train=prepare_dict(df_train)
dict_val=prepare_dict(df_val)

In [21]:
with mlflow.start_run():
    params=dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=13)
    mlflow.log_params(params)
    
    dv=DictVectorizer()
    model=RandomForestRegressor(**params, n_jobs=-1)
    
    X_train= dv.fit_transform(dict_train)
    model.fit(X_train, y_train)
    
    X_val= dv.transform(dict_val)
    y_pred=model.predict(X_val)
    
    rmse=mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric("rmse", rmse)

    #os.makedirs("preprocessor", exist_ok=True)
    with open("dict_vectorizer.bin", 'wb') as f_out:
        pickle.dump(dv, f_out)
        
    mlflow.log_artifact("dict_vectorizer.bin", artifact_path="preprocessor")
    os.makedirs("model_1", exist_ok=True)
    mlflow.sklearn.log_model(model, name="model_1/model")
    
    

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 13} 6.756305038881697
🏃 View run respected-duck-117 at: http://127.0.0.1:5000/#/experiments/2/runs/bd66d2ec3f154ce8bc89dde038c1fc2b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


RestException: INVALID_PARAMETER_VALUE: Invalid model name ('model_1/model') provided. Model name must be a non-empty string and cannot contain the following characters: ('/', ':', '.', '%', '"', "'")